In [1]:
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Load the uploaded dataset
file_path = 'E:\Economic_Data\Input data\merged-dataset\merged_datasets.csv'
merged_data = pd.read_csv(file_path, parse_dates=['datetime'])


# Function to clean and convert values to numeric
def clean_and_convert(value):
    if isinstance(value, str):
        # Remove non-numeric characters except for necessary ones like '.' and '-'
        value = re.sub(r'[^\d.-]', '', value)
        # Convert to float
        try:
            return float(value)
        except ValueError:
            return None
    return value

# Apply the function to the 'Actual', 'Forecast', and 'Previous' columns
merged_data['Actual'] = merged_data['Actual'].apply(clean_and_convert)
merged_data['Forecast'] = merged_data['Forecast'].apply(clean_and_convert)
merged_data['Previous'] = merged_data['Previous'].apply(clean_and_convert)

# Calculate differences for all events
merged_data['Actual_Forecast_Diff'] = merged_data['Actual'] - merged_data['Forecast']
merged_data['Actual_Previous_Diff'] = merged_data['Actual'] - merged_data['Previous']

# Fill missing values in the differences with zero
merged_data['Actual_Forecast_Diff'].fillna(0, inplace=True)
merged_data['Actual_Previous_Diff'].fillna(0, inplace=True)

# Check for any remaining NaNs
print("Number of NaNs in Actual_Forecast_Diff:", merged_data['Actual_Forecast_Diff'].isna().sum())
print("Number of NaNs in Actual_Previous_Diff:", merged_data['Actual_Previous_Diff'].isna().sum())

# Pivot the data to get differences for all events
pivot_data_forecast = merged_data.pivot(index='datetime', columns='Event', values='Actual_Forecast_Diff')
pivot_data_previous = merged_data.pivot(index='datetime', columns='Event', values='Actual_Previous_Diff')

# Fill missing values in pivot tables with zeros
pivot_data_forecast.fillna(0, inplace=True)
pivot_data_previous.fillna(0, inplace=True)

# Concatenate the pivoted data to have all differences in a single DataFrame
all_differences = pd.concat([pivot_data_forecast, pivot_data_previous], axis=1, keys=['Forecast', 'Previous'])

Number of NaNs in Actual_Forecast_Diff: 0
Number of NaNs in Actual_Previous_Diff: 0


C:\Users\Zeinab\AppData\Local\Temp\ipykernel_4908\2165847126.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['Actual_Forecast_Diff'].fillna(0, inplace=True)
C:\Users\Zeinab\AppData\Local\Temp\ipykernel_4908\2165847126.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

In [2]:
# Flatten the multi-index columns
all_differences.columns = [f'{lvl1}_{lvl2}' for lvl1, lvl2 in all_differences.columns]

# Standardize the data
scaler = StandardScaler()
all_differences_scaled = scaler.fit_transform(all_differences)

# Calculate the correlation matrix
correlation_matrix = all_differences.corr()

# Identify events with high correlation (greater than 0.7 or less than -0.7)
high_corr_pairs = []
for i in correlation_matrix.columns:
    for j in correlation_matrix.columns:
        if i != j and abs(correlation_matrix.at[i, j]) >= 0.7:
            high_corr_pairs.append((i, j, correlation_matrix.at[i, j]))

# Create a DataFrame for high correlation pairs
high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Event_1', 'Event_2', 'Correlation'])

# Save to CSV
output_path = 'high_correlation_events.csv'
high_corr_df.to_csv(output_path, index=False)

print(f"High correlation events saved to {output_path}")
print(high_corr_df.head())

High correlation events saved to high_correlation_events.csv
                                        Event_1  \
0  Forecast_ADP Nonfarm Employment Change (Apr)   
1  Forecast_ADP Nonfarm Employment Change (Aug)   
2  Forecast_ADP Nonfarm Employment Change (Dec)   
3  Forecast_ADP Nonfarm Employment Change (Feb)   
4  Forecast_ADP Nonfarm Employment Change (Jan)   

                                        Event_2  Correlation  
0  Previous_ADP Nonfarm Employment Change (Apr)     1.000000  
1  Previous_ADP Nonfarm Employment Change (Aug)     1.000000  
2  Previous_ADP Nonfarm Employment Change (Dec)     0.939830  
3  Previous_ADP Nonfarm Employment Change (Feb)     0.903623  
4  Previous_ADP Nonfarm Employment Change (Jan)     0.988509  


In [3]:
from datetime import datetime, timedelta

# Load the uploaded dataset
file_path = 'E:\Economic_Data\Input data\price.csv'
btc_price_data = pd.read_csv(file_path, parse_dates=['datetime'])

# Helper function to calculate BTC price change at given intervals
def calculate_price_change(event_time, interval_minutes):
    end_time = event_time + timedelta(minutes=interval_minutes)
    start_price = btc_price_data.loc[btc_price_data['datetime'] == event_time, 'close']
    end_price = btc_price_data.loc[btc_price_data['datetime'] == end_time, 'close']
    if not start_price.empty and not end_price.empty:
        price_change = ((end_price.values[0] - start_price.values[0]) / start_price.values[0]) * 100
        return price_change
    return None

# Calculate price changes at different intervals
intervals = [5, 15, 30, 60]
for interval in intervals:
    merged_data[f'price_change_{interval}min'] = merged_data['datetime'].apply(lambda x: calculate_price_change(x, interval))


In [4]:
merged_data

,Cur.,Imp.,Event,Actual,Forecast,Previous,datetime,Actual_Forecast_Diff,Actual_Previous_Diff,price_change_5min,price_change_15min,price_change_30min,price_change_60min
0,USD,3.0,S&P Global US Manufacturing PMI (Dec),46.200,46.2,47.70,2023-01-03 14:45:00,0.000,-1.500,-0.160049,-0.477757,-0.502840,-0.591822
1,USD,2.0,Construction Spending (MoM) (Nov),0.200,-0.4,-0.20,2023-01-03 15:00:00,0.600,0.400,-0.022202,-0.025203,-0.265228,0.002400
2,USD,1.0,3-Month Bill Auction,4.410,NaN,4.35,2023-01-03 16:30:00,0.000,0.060,-0.000601,0.040873,-0.140048,-0.119011
3,USD,1.0,6-Month Bill Auction,4.640,NaN,4.60,2023-01-03 16:30:00,0.000,0.040,-0.000601,0.040873,-0.140048,-0.119011
4,USD,NaN,MBA Mortgage Applications (WoW),-10.300,NaN,0.90,2023-01-04 12:00:00,0.000,-11.200,0.005942,0.030303,-0.009507,-0.152109
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5637,USD,1.0,OPEC Crude oil Production UAE (Barrel),2.920,NaN,2.92,2024-04-30 16:00:00,0.000,0.000,0.063985,-0.222715,-0.877702,-0.910764
5638,USD,1.0,OPEC Crude oil Production Iran (Barrel),3.150,NaN,3.03,2024-04-30 16:00:00,0.000,0.120,0.063985,-0.222715,-0.877702,-0.910764
5639,USD,1.0,OPEC Crude oil Production Iraq (Barrel),4.150,NaN,4.13,2024-04-30 16:00:00,0.000,0.020,0.063985,-0.222715,-0.877702,-0.910764
5640,USD,1.0,OPEC Crude oil Production Saudi Arabia (Barrel),9.000,NaN,8.98,2024-04-30 16:00:00,0.000,0.020,0.063985,-0.222715,-0.877702,-0.910764


In [5]:
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf

# Apply PCA
pca = PCA(n_components=0.95)  # Retain 95% of variance
all_differences_pca = pca.fit_transform(all_differences_scaled)

# Convert PCA components to DataFrame
pca_df = pd.DataFrame(all_differences_pca, index=all_differences.index)

# Merge PCA components with the original merged_data for modeling
merged_data_pca = merged_data[['datetime', 'Event']].merge(pca_df, left_index=True, right_index=True)

# Fit the hierarchical model with PCA components
formula = 'Pct_Change_5min ~ ' + ' + '.join(pca_df.columns.astype(str))
model = smf.mixedlm(formula, data=merged_data_pca, groups=merged_data_pca['Event'])
result = model.fit()

# Display the summary
print(result.summary())

PatsyError: numbers besides '0' and '1' are only allowed with **
    Pct_Change_5min ~ 0 + 1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10 + 11 + 12 + 13 + 14 + 15 + 16 + 17 + 18 + 19 + 20 + 21 + 22 + 23 + 24 + 25 + 26 + 27 + 28 + 29 + 30 + 31 + 32 + 33 + 34 + 35 + 36 + 37 + 38 + 39 + 40 + 41 + 42 + 43 + 44 + 45 + 46 + 47 + 48 + 49 + 50 + 51 + 52 + 53 + 54 + 55 + 56 + 57 + 58 + 59 + 60 + 61 + 62 + 63 + 64 + 65 + 66 + 67 + 68 + 69 + 70 + 71 + 72 + 73 + 74 + 75 + 76 + 77 + 78 + 79 + 80 + 81 + 82 + 83 + 84 + 85 + 86 + 87 + 88 + 89 + 90 + 91 + 92 + 93 + 94 + 95 + 96 + 97 + 98 + 99 + 100 + 101 + 102 + 103 + 104 + 105 + 106 + 107 + 108 + 109 + 110 + 111 + 112 + 113 + 114 + 115 + 116 + 117 + 118 + 119 + 120 + 121 + 122 + 123 + 124 + 125 + 126 + 127 + 128 + 129 + 130 + 131 + 132 + 133 + 134 + 135 + 136 + 137 + 138 + 139 + 140 + 141 + 142 + 143 + 144 + 145 + 146 + 147 + 148 + 149 + 150 + 151 + 152 + 153 + 154 + 155 + 156 + 157 + 158 + 159 + 160 + 161 + 162 + 163 + 164 + 165 + 166 + 167 + 168 + 169 + 170